[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IyadSultan/CCI/blob/main/session4/solutions/Lab3_Radiology_Summarization_Solutions.ipynb)

# Lab 3: Fine-Tuning for Radiology Report Summarization
## CCI Session 4 — Cancer Care Informatics · KHCC AI Office

---

## ⚠️ Before You Start
**Enable GPU:** Go to `Runtime → Change runtime type → T4 GPU`

---

## Clinical Scenario

Every radiology report at KHCC has two sections:

- **Findings** — the detailed technical description of what the radiologist observed. Can be 100–300 words. Written for completeness.
- **Impression** — the 1–3 sentence clinical conclusion. The answer to "what does this mean for the patient?"

Writing the impression requires synthesizing findings into a concise clinical judgment. In a busy center reviewing 200+ chest X-rays per day, automating impression generation — even partially — could meaningfully reduce cognitive load.

In this lab, you will:
1. Load a real dataset of radiology reports (MIMIC-CXR)
2. Test a general-purpose language model *before* any training (baseline)
3. Fine-tune the model on radiology-specific data
4. Compare before vs. after using ROUGE metrics

---

## Key Concept: What Is Fine-Tuning?

A pre-trained language model like `flan-t5-small` has already learned from billions of words of general text. It knows English grammar, common medical terms, and summarization patterns — but it has never specifically learned that **radiology findings lead to radiology impressions**.

**Fine-tuning** continues training the model on your specific task. Instead of learning from scratch (which requires massive data and compute), you gently adjust the model's existing knowledge to fit your domain.

Think of it as a residency rotation after medical school: the doctor already has foundational knowledge, and the rotation specializes them for a specific clinical context.

```
Pre-trained model (general English)  →  Fine-tuned model (radiology summarization)
        flan-t5-small                →  flan-t5-small + your radiology data
```

---
## Cell 0: Setup — Install Libraries and Verify GPU

In [ ]:
# Install required libraries:
# - transformers : Hugging Face library for pre-trained models
# - datasets     : Hugging Face library for loading datasets
# - evaluate     : Hugging Face library for computing metrics (ROUGE, BLEU, etc.)
# - rouge_score  : Backend that `evaluate` uses to compute ROUGE
# - accelerate   : Speeds up training on GPU hardware
!pip install -q transformers datasets evaluate rouge_score accelerate

import torch

print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU model:     {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device:  {device}")

if not torch.cuda.is_available():
    print("\n⚠️  WARNING: No GPU detected. Training will be very slow.")
    print("   Go to Runtime → Change runtime type → Select T4 GPU")

---
## Cell 1: Load the Radiology Dataset

We use **MIMIC-CXR** — a de-identified dataset of chest X-ray reports from Beth Israel Deaconess Medical Center (Boston, USA). It contains ~227,000 reports, each with a `findings` field and an `impression` field.

For training efficiency, we take a **random subset of 1,000 reports**. This is small by ML standards, but enough to demonstrate the fine-tuning effect within a lab session.

> **Why 1,000?** Training on 1,000 examples takes ~5–8 minutes on a T4 GPU. The full dataset would take hours. In a real deployment at KHCC, you would use all available data — ideally combined with your own institutional reports.

In [ ]:
from datasets import load_dataset

# Load the full dataset from Hugging Face Hub
ds = load_dataset("tgrex6/mimic-cxr-reports-summarization")
print(ds)
print()

# Preview the first 3 reports so you can see the input/output structure
print("=" * 60)
print("SAMPLE REPORTS (truncated for display)")
print("=" * 60)
for i in range(3):
    print(f"\n--- Report {i+1} ---")
    print(f"FINDINGS (input):    {ds['train'][i]['findings'][:250]}...")
    print(f"IMPRESSION (target): {ds['train'][i]['impression']}")

# Shuffle and take 1,000 examples for this lab
# seed=42 ensures everyone gets the same subset (reproducibility)
small_ds = ds["train"].shuffle(seed=42).select(range(1000))
print(f"\nTraining subset size: {len(small_ds)} reports")

---
## Cell 2: Explore the Data

Before training any model, always understand your data. Key questions:
- How long are the inputs (findings) vs. the outputs (impressions)?
- Are there empty or corrupt records that could break training?
- What is the compression ratio — how much does the impression shorten the findings?

This informs the `max_length` parameters you will set during tokenization.

In [ ]:
# Measure lengths in words (approximate — tokenizers use subword units, not whole words)
findings_lengths   = [len(x["findings"].split())   for x in small_ds]
impression_lengths = [len(x["impression"].split()) for x in small_ds]

avg_findings   = sum(findings_lengths)   / len(findings_lengths)
avg_impression = sum(impression_lengths) / len(impression_lengths)
compression    = avg_findings / avg_impression

print("=" * 40)
print("DATASET STATISTICS")
print("=" * 40)
print(f"Avg findings length:   {avg_findings:.0f} words")
print(f"Avg impression length: {avg_impression:.0f} words")
print(f"Compression ratio:     {compression:.1f}x")
print()
print(f"Impressions are {compression:.0f}x shorter than findings on average.")
print("The model must learn to identify and extract the clinically important parts.")

# Check for empty records — these cause NaN loss during training
empty_findings    = sum(1 for x in small_ds if not x["findings"].strip())
empty_impressions = sum(1 for x in small_ds if not x["impression"].strip())
print(f"\nEmpty findings:    {empty_findings}")
print(f"Empty impressions: {empty_impressions}")

# Filter out any empty rows before training
small_ds = small_ds.filter(
    lambda x: len(x["findings"].strip()) > 0 and len(x["impression"].strip()) > 0
)
print(f"\nClean dataset size: {len(small_ds)} reports (after filtering)")

---
## Cell 3: Load the Model and Tokenizer

We use **`google/flan-t5-small`** — a sequence-to-sequence (seq2seq) model trained by Google on 1,800+ language tasks with natural language instructions.

### Why flan-t5-small?

| Property | Value |
|---|---|
| Parameters | ~80 million |
| Architecture | Encoder-Decoder (T5) |
| Training | Instruction fine-tuned on diverse NLP tasks |
| Size on disk | ~308 MB |
| Why suitable | Designed for text-to-text tasks; responds well to prompts like "Summarize the following..." |

### Tokenizer vs. Model

- **Tokenizer**: Converts text → token IDs (numbers). Every model has its own vocabulary (~32,000 tokens for flan-t5).
- **Model**: Takes token IDs as input and produces output token IDs.

Both must always come from the same checkpoint to be compatible.

### ⚠️ Why `force_download=True`?

This flag bypasses the local Hugging Face cache and downloads fresh weights from the Hub. We use it to avoid accidentally loading a previously saved (possibly corrupted or overfit) checkpoint. Without fresh weights, the model loss can start at 0.0 and never learn anything meaningful.

In [ ]:
import os, shutil
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Clean up any previous checkpoint folders from prior runs
for folder in ["./radiology-summarizer", "./radiology-summarizer-final"]:
    if os.path.exists(folder):
        shutil.rmtree(folder)
        print(f"Cleaned up: {folder}")

# Clear the Hugging Face model cache to ensure fresh weights
hf_cache = os.path.expanduser("~/.cache/huggingface/hub/models--google--flan-t5-small")
if os.path.exists(hf_cache):
    shutil.rmtree(hf_cache)
    print("Cleaned up HuggingFace model cache")

model_name = "google/flan-t5-small"

# Load tokenizer — converts between text and token IDs
tokenizer = AutoTokenizer.from_pretrained(model_name, force_download=True)

# Load model with explicit float32 dtype
# float16 (fp16) can cause NaN loss with this checkpoint due to tied embedding weights
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    force_download=True,
    dtype=torch.float32,
).to(device)

print(f"\nModel loaded on: {device}")
print(f"Model parameters: {model.num_parameters():,}")

# Sanity check: a fresh model should produce real (non-NaN) logits
model.eval()
dummy = torch.ones(1, 10, dtype=torch.long).to(device)
with torch.no_grad():
    out = model(input_ids=dummy, decoder_input_ids=dummy)
has_nan = out.logits.isnan().any().item()
print(f"\nSanity check — logits contain NaN: {has_nan}")
if has_nan:
    print("⚠️  Model weights appear corrupted. Try: Runtime → Disconnect and delete runtime")
else:
    print("✅ Model weights are healthy. Proceed to Cell 4.")

---
## Cell 4: Baseline — How Does the Model Perform *Before* Training?

Before fine-tuning, we test the model on 5 radiology reports to establish a **baseline**. This step is critical — without a baseline, you cannot measure whether fine-tuning helped.

### What to Expect at Baseline

The model has never seen radiology reports in a structured task-specific way. It will attempt to summarize using general language patterns, but typically:
- Uses generic, non-clinical phrases ("Observations are indicated")
- Misses specific findings (effusions, opacities, vertebral fractures)
- Sometimes hallucinates non-existent medical terms

### The Prompt Prefix

We prefix every input with `"Summarize the following radiology findings:\n"`. This instructs the flan-t5 model (which was fine-tuned on instruction-following tasks) to treat the text as something to summarize. Without a prefix, the model may simply continue the text instead of summarizing it.

In [ ]:
import evaluate

rouge = evaluate.load("rouge")

# This prefix tells flan-t5 what task we want it to perform
PREFIX = "Summarize the following radiology findings:\n"

# Use the first 5 reports for a consistent before/after comparison
test_samples = small_ds.select(range(5))
actual_impressions = [s["impression"] for s in test_samples]

print("=" * 60)
print("BASELINE: MODEL OUTPUT BEFORE FINE-TUNING")
print("=" * 60)

baseline_predictions = []
model.eval()

for i, sample in enumerate(test_samples):
    input_text = PREFIX + sample["findings"]

    # Tokenize: convert text to token IDs
    # max_length=512: truncate findings longer than 512 tokens
    # truncation=True: silently cut off text exceeding max_length
    inputs = tokenizer(
        input_text,
        return_tensors="pt",   # Return PyTorch tensors (not lists)
        max_length=512,
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,   # Max length of generated impression
            num_beams=4,          # Beam search: explore 4 candidate sequences simultaneously
            early_stopping=True   # Stop when all beams produce an end token
        )

    # Decode: convert output token IDs back to readable text
    prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)
    baseline_predictions.append(prediction)

    print(f"\n--- Report {i+1} ---")
    print(f"ACTUAL:    {sample['impression'][:220]}")
    print(f"PREDICTED: {prediction[:220]}")

# Compute ROUGE scores for the baseline model
baseline_results = rouge.compute(
    predictions=baseline_predictions,
    references=actual_impressions
)

print("\n" + "=" * 40)
print("BASELINE ROUGE SCORES")
print("=" * 40)
for key, value in baseline_results.items():
    print(f"{key}: {value:.4f}")
print("\n→ Save these numbers. You will compare them after fine-tuning.")

---
## Cell 5: Preprocessing — Tokenize the Dataset for Training

Before training, we convert all text into token IDs and format them correctly for the seq2seq model.

### Key Concepts in This Cell

#### `max_length` for inputs vs. labels
- **512 tokens for findings (input)**: Covers most radiology reports. Longer reports are truncated from the end.
- **128 tokens for impressions (labels)**: Most impressions are 20–60 words. 128 tokens is comfortably sufficient.

These numbers come from the data exploration in Cell 2.

#### `padding="max_length"`
Pads shorter sequences with a special `[PAD]` token until they reach the fixed length. Without padding, examples of different lengths cannot be batched together — and batching is essential for GPU efficiency.

#### The `-100` label masking trick (critical)
After padding, the label tensor contains real token IDs mixed with padding token IDs. We replace all padding positions with `-100`. Why?

PyTorch's cross-entropy loss function has a built-in `ignore_index=-100` parameter. Any position in the labels set to `-100` is **completely excluded from the loss calculation**. This means the model is only trained to predict real impression tokens — not to predict padding.

Without this fix, the loss becomes `NaN` or collapses to `0.0` in the first training step, because the model either correctly predicts padding everywhere or gets an undefined gradient.

#### Train/Validation Split
We split 85%/15% (850 train, 150 validation). The validation set serves as a held-out check during training — if validation loss starts rising while training loss keeps falling, we know the model is overfitting.

In [ ]:
def preprocess_function(examples):
    """
    Tokenize findings (input) and impressions (target labels) for seq2seq training.

    Input:  {'findings': [...], 'impression': [...]}
    Output: {'input_ids': [...], 'attention_mask': [...], 'labels': [...]}
    """
    # Add the task instruction prefix to every findings text
    inputs = [PREFIX + finding for finding in examples["findings"]]

    # Tokenize the inputs (findings → encoder input)
    model_inputs = tokenizer(
        inputs,
        max_length=512,       # Truncate findings beyond 512 tokens
        truncation=True,
        padding="max_length", # Pad shorter findings to exactly 512 tokens
    )

    # Tokenize the labels (impressions → decoder target)
    labels = tokenizer(
        examples["impression"],
        max_length=128,       # Impressions are short; 128 tokens is sufficient
        truncation=True,
        padding="max_length",
    )

    # Replace padding token IDs with -100 so PyTorch ignores them in the loss
    # Without this: loss is NaN (undefined) or collapses to 0 immediately
    label_ids = labels["input_ids"]
    label_ids = [
        [(token if token != tokenizer.pad_token_id else -100) for token in label]
        for label in label_ids
    ]

    model_inputs["labels"] = label_ids
    return model_inputs


# Create train/validation split (85% train, 15% validation)
split = small_ds.train_test_split(test_size=0.15, seed=42)
train_data = split["train"]    # ~850 reports
val_data   = split["test"]     # ~150 reports

print(f"Train split: {len(train_data)} reports")
print(f"Val split:   {len(val_data)} reports")
print()

# Apply tokenization to both splits
# batched=True: processes multiple examples at once (faster than one-by-one)
# remove_columns: drop original text columns (model only needs token IDs)
tokenized_train = train_data.map(
    preprocess_function, batched=True, remove_columns=train_data.column_names
)
tokenized_val = val_data.map(
    preprocess_function, batched=True, remove_columns=val_data.column_names
)

# Convert to PyTorch tensor format for the DataLoader
tokenized_train.set_format("torch")
tokenized_val.set_format("torch")

print(f"Tokenized features: {list(tokenized_train.features.keys())}")
print(f"Input shape:        {tokenized_train[0]['input_ids'].shape}  (512 tokens)")
print(f"Label shape:        {tokenized_train[0]['labels'].shape}   (128 tokens)")

# Confirm -100 masking is working correctly
sample_labels = tokenized_train[0]['labels']
n_real = (sample_labels != -100).sum().item()
n_pad  = (sample_labels == -100).sum().item()
print(f"\nLabel check on first example:")
print(f"  Real tokens (will be learned): {n_real}")
print(f"  Pad tokens  (will be ignored): {n_pad}")
print("  If real tokens = 0, the -100 masking is incorrectly applied to everything.")

---
## Cell 6: Sanity Check — Verify Loss Before Training

**Always run this before a full training run.**

A freshly loaded model should have a high loss (typically 5–10) because it is essentially guessing randomly among 32,000 vocabulary tokens. If loss is:

- **0.0** → weights are from a previously overfit or broken checkpoint
- **NaN** → weight corruption, dtype mismatch, or all labels are -100
- **5–10** → healthy starting point, proceed with training

This step costs only a few seconds and can save you from wasting 10+ minutes on a broken run.

In [ ]:
from torch.utils.data import DataLoader

# Load one mini-batch of 4 examples
loader = DataLoader(tokenized_train, batch_size=4, shuffle=True)
batch = next(iter(loader))
batch = {k: v.to(device) for k, v in batch.items()}

# Single forward pass — no weight updates (torch.no_grad)
model.train()
with torch.no_grad():
    outputs = model(**batch)

loss_val = outputs.loss.item()
print(f"Initial loss on one batch: {loss_val:.4f}")
print()

if loss_val == 0.0:
    print("⛔ STOP: Loss is 0.")
    print("   The model likely loaded from a previously overfit checkpoint.")
    print("   Re-run Cell 3 with force_download=True to reload fresh weights.")
elif loss_val != loss_val:  # NaN != NaN is always True in IEEE 754
    print("⛔ STOP: Loss is NaN.")
    print("   Possible causes: corrupted weights, all labels are -100, dtype mismatch.")
    print("   Try: Runtime → Disconnect and delete runtime → Reconnect → Run from top.")
elif loss_val >= 3.0:
    print("✅ Loss looks healthy — the model is genuinely uncertain.")
    print(f"   A loss of ~{loss_val:.1f} means near-random prediction, which is expected before training.")
    print("   Proceed to Cell 7 to configure training.")
else:
    print("⚠️  Loss is lower than expected for a fresh model.")
    print("   Verify that weights are loaded from the base checkpoint, not a fine-tuned one.")

---
## Cell 7: Training Configuration — Hyperparameters Explained

**Hyperparameters** are the settings you choose before training begins. Unlike model weights (which the optimizer learns automatically), hyperparameters are set by the practitioner. Choosing them well is both science and informed judgment.

---

### 🔧 Every Hyperparameter Explained

#### `num_train_epochs = 5`
One **epoch** = one complete pass through all training examples.
With 850 examples and batch size 8: 850 ÷ 8 = ~107 steps/epoch → 535 total steps.

- **Too few** → underfitting: model hasn't learned enough from the data
- **Too many** → overfitting: model memorizes training examples, generalizes poorly
- **How to choose**: monitor validation loss — stop when it plateaus or starts rising

---

#### `per_device_train_batch_size = 8`
How many examples are processed together in one forward + backward pass.

- **Larger batch** (16, 32) → more stable gradient estimates, but uses more GPU memory; may need higher learning rate
- **Smaller batch** (2, 4) → noisier gradients, but sometimes helps escape flat regions of the loss surface
- **T4 GPU limit**: With 512-token inputs in float32, batch size 8 fits comfortably in 16 GB VRAM

---

#### `learning_rate = 3e-5`
The most important hyperparameter. Controls the step size the optimizer takes after each batch.

- **Too high** (e.g., `5e-4`): Loss collapses to 0 at step 1 — the model "overshoots" and destroys what it learned during pre-training
- **Too low** (e.g., `1e-8`): Training makes negligible progress even after many epochs
- **Fine-tuning rule of thumb**: Always use a smaller LR than was used for pre-training. For T5: `1e-5` to `5e-5` is typical.
- **Diagnostic**: If loss hits 0 at step 1, reduce LR by 10×

---

#### `weight_decay = 0.05`
A regularization technique that penalizes large weight values. Helps prevent overfitting.

Mathematically: adds `weight_decay × sum(w²)` to the loss at each step, discouraging the model from relying too heavily on any individual parameter.

- Particularly important with small datasets (850 examples)
- Typical range: `0.01` to `0.1`

---

#### `warmup_steps = 50`
For the first 50 steps, the learning rate gradually increases from 0 to `learning_rate`. After that, it follows the normal schedule (usually linear decay).

- **Why**: Jumping immediately to the full LR in the first batch can cause unstable, oversized updates that corrupt the pre-trained weights
- **Rule of thumb**: 5–10% of total steps (535 × 0.1 ≈ 53)

---

#### `eval_strategy = "epoch"` + `load_best_model_at_end = True`
After each epoch, the Trainer evaluates on the validation set and records the validation loss. At the end of training, the checkpoint with the lowest validation loss is automatically selected.

This prevents using an overfit model from the final epoch — if the best performance was at epoch 3 out of 5, you get epoch 3's weights automatically.

---

#### `fp16 = False`
Mixed-precision training (float16) is normally a free speedup on T4 GPUs. However, the flan-t5-small checkpoint has tied embedding weights (`shared.weight` and `lm_head.weight`) that cause numerical instability in float16, producing NaN loss from the very first step.

Disabling fp16 costs ~30% training speed, but guarantees numerical stability. In production with a properly configured checkpoint, fp16 is safe to enable.

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

training_args = TrainingArguments(
    output_dir="./radiology-summarizer-final",

    # --- How long to train ---
    num_train_epochs=5,              # 5 complete passes through the training data
                                     # Too few → underfitting | Too many → overfitting

    # --- Batch sizes ---
    per_device_train_batch_size=8,   # 8 examples per training step
    per_device_eval_batch_size=8,    # 8 examples per evaluation step

    # --- Learning rate ---
    learning_rate=3e-5,              # Conservative LR for fine-tuning (not pre-training)
                                     # If loss collapses to 0 at step 1, reduce this by 10x

    # --- Regularization ---
    weight_decay=0.05,               # L2 penalty — important with only 850 training examples
    warmup_steps=50,                 # Gradually ramp up LR for the first 50 steps

    # --- Logging ---
    logging_steps=25,                # Print training loss every 25 steps

    # --- Validation and checkpointing ---
    eval_strategy="epoch",           # Evaluate on validation set after each epoch
    save_strategy="epoch",           # Save a checkpoint after each epoch
    load_best_model_at_end=True,     # Use the checkpoint with lowest val loss at the end
    metric_for_best_model="eval_loss",

    # --- Precision ---
    fp16=False,                      # Disabled: flan-t5 checkpoint causes NaN with fp16

    report_to="none",                # Disable Weights & Biases / MLflow logging
)

# DataCollatorForSeq2Seq handles dynamic padding within each batch
# pad_to_multiple_of=8: aligns sequence lengths to multiples of 8 for GPU efficiency
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    pad_to_multiple_of=8
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,     # Validation set for monitoring generalization
    data_collator=data_collator,
)

# Show the training plan so you know what to expect
steps_per_epoch = len(tokenized_train) // training_args.per_device_train_batch_size
total_steps = steps_per_epoch * training_args.num_train_epochs

print("=" * 40)
print("TRAINING PLAN")
print("=" * 40)
print(f"Training examples:   {len(tokenized_train)}")
print(f"Batch size:          {training_args.per_device_train_batch_size}")
print(f"Steps per epoch:     {steps_per_epoch}")
print(f"Total epochs:        {training_args.num_train_epochs}")
print(f"Total steps:         {total_steps}")
print(f"Learning rate:       {training_args.learning_rate}")
print(f"Warmup steps:        {training_args.warmup_steps}")
print(f"Weight decay:        {training_args.weight_decay}")
print(f"fp16:                {training_args.fp16}")
print()
print("Trainer configured. Run Cell 8 to start training.")

---
## Cell 8: Fine-Tune the Model

This is the core training step. On a T4 GPU, expect **5–8 minutes** for 5 epochs on 850 examples.

### What to Watch in the Loss Table

The Trainer prints a table after each epoch:

| Epoch | Training Loss | Validation Loss |
|---|---|---|
| 1 | high | high |
| 2 | lower | lower |
| ... | ... | ... |

**Healthy training:**
- Both losses decrease across epochs
- Validation loss stays close to training loss (no large gap = no overfitting)
- Loss never hits 0.0 from step 1

**Warning signs:**
| Pattern | Likely Cause |
|---|---|
| Training loss = 0.0 at step 1 | Broken/overfit weights loaded — re-run Cell 3 |
| Validation loss = NaN | Label -100 masking not applied — re-run Cell 5 |
| Val loss rises while train loss falls | Overfitting — reduce epochs or increase weight_decay |

Note: The absolute loss values will be high (7–10) because we are using only 850 examples and a small model. The **downward trend** matters more than the absolute number.

In [ ]:
print("Starting fine-tuning...")
print("Expected time: 5-8 minutes on T4 GPU")
print("Watch: training loss and validation loss should both decrease each epoch")
print()

train_result = trainer.train()

print("\n" + "=" * 40)
print("TRAINING COMPLETE")
print("=" * 40)
print(f"Total training time:   {train_result.metrics['train_runtime']:.0f} seconds")
print(f"Final average loss:    {train_result.metrics['train_loss']:.4f}")
print(f"Throughput:            {train_result.metrics['train_samples_per_second']:.1f} samples/sec")
print()
print("The model with the best validation loss has been automatically loaded.")

---
## Cell 9: After Fine-Tuning — Qualitative Comparison

Now we run the same 5 test reports through the fine-tuned model and compare three outputs side-by-side:
1. **ACTUAL** — what the radiologist wrote
2. **BEFORE TUNING** — what the baseline model generated
3. **AFTER TUNING** — what the fine-tuned model generates

Read these carefully. ROUGE gives you a number, but reading the actual text tells you whether the model is producing *clinically coherent* language. Specific things to look for:

- Did hallucinated words disappear?
- Is the model now using proper radiology terminology?
- Does the structure resemble a real impression (declarative, concise)?

In [ ]:
print("=" * 60)
print("QUALITATIVE COMPARISON: BEFORE vs. AFTER FINE-TUNING")
print("=" * 60)

finetuned_predictions = []
model.eval()

for i, sample in enumerate(test_samples):
    input_text = PREFIX + sample["findings"]
    inputs = tokenizer(
        input_text, return_tensors="pt", max_length=512, truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            num_beams=4,
            early_stopping=True
        )

    prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)
    finetuned_predictions.append(prediction)

    print(f"\n--- Report {i+1} ---")
    print(f"ACTUAL:        {sample['impression'][:220]}")
    print(f"BEFORE TUNING: {baseline_predictions[i][:220]}")
    print(f"AFTER TUNING:  {prediction[:220]}")

# Compute ROUGE scores for the fine-tuned model
finetuned_results = rouge.compute(
    predictions=finetuned_predictions,
    references=actual_impressions,
)
print("\nROUGE scores computed. Run Cell 10 to see the comparison table.")

---
## Cell 10: ROUGE Metrics — What Do They Actually Measure?

**ROUGE** (Recall-Oriented Understudy for Gisting Evaluation) is the standard evaluation metric for summarization. It measures word overlap between the model's output and a reference (human-written) summary.

---

### The Four ROUGE Variants

#### ROUGE-1: Unigram (single word) overlap

Counts individual words that appear in both the prediction and the reference.

```
Reference:  "pleural effusion and pulmonary edema"
Prediction: "small pleural effusion"

Matching words: pleural, effusion  → ROUGE-1 = 2 matches / 4 reference words = 0.50
```

Captures: *Does the model use the right vocabulary?*
Limitation: "No edema" and "edema" both match on the word "edema" — polarity is ignored.

---

#### ROUGE-2: Bigram (two consecutive words) overlap

Counts matching pairs of consecutive words.

```
Reference:  "bilateral pleural effusion"
Prediction: "small bilateral effusion"

Reference bigrams:  (bilateral pleural), (pleural effusion)
Prediction bigrams: (small bilateral), (bilateral effusion)
Matching bigrams:   none! → ROUGE-2 = 0.0
```

Stricter than ROUGE-1. **Improvement in ROUGE-2 is the strongest signal that the model has learned domain-specific phrases**, not just individual words. A model that learns to say "pleural effusion" (a bigram) scores higher than one that knows "pleural" and "effusion" separately.

---

#### ROUGE-L: Longest Common Subsequence (LCS)

Finds the longest sequence of words that appears in the same order in both texts, but not necessarily consecutively.

```
Reference:  "no acute cardiopulmonary abnormality"
Prediction: "no acute pulmonary abnormality"

LCS: "no acute _ abnormality" (3 words in correct order)
ROUGE-L = 3 / 4 = 0.75
```

Captures: *Is the sentence structure and word order correct?*
A model that reverses the word order ("abnormality cardiopulmonary no acute") scores low on ROUGE-L even if all words are present.

---

#### ROUGE-Lsum: Sentence-level ROUGE-L

Same as ROUGE-L, but computed sentence by sentence and then averaged. Better for multi-sentence impressions where each sentence should independently match the reference.

---

### Interpreting ROUGE Scores in Clinical NLP

| ROUGE Score | Interpretation for Radiology |
|---|---|
| 0.00–0.10 | Very poor — model generates unrelated or hallucinated text |
| 0.10–0.25 | Poor — uses some radiology vocabulary but wrong structure |
| 0.25–0.40 | Moderate — captures key findings, misses detail or phrasing |
| 0.40–0.60 | Good — competitive with human summarization |
| > 0.60 | Excellent — approaches radiologist-level performance |

**Expected for this lab**: With 850 training examples and an 80M-parameter model, ROUGE-1 scores of 0.15–0.35 represent meaningful improvement over the baseline and are realistic given the data constraints.

---

### Critical Limitations of ROUGE

1. **ROUGE measures overlap, not correctness.** A model that says "no pneumonia" when the actual impression says "pneumonia" may score 0.5 ROUGE on the word "pneumonia" — but the clinical meaning is opposite.

2. **Synonyms are penalized.** "Fluid around the lung" and "pleural effusion" mean the same thing clinically, but ROUGE scores them as 0 overlap.

3. **ROUGE cannot replace radiologist evaluation.** In a clinical deployment at KHCC, ROUGE would be used as a first-pass filter, followed by a formal blinded radiologist rating study.

4. **ROUGE-2 improvement is the most meaningful.** If ROUGE-2 improves after fine-tuning, it means the model has learned radiology-specific two-word phrases — a reliable indicator of domain adaptation.

In [ ]:
print("=" * 58)
print("ROUGE SCORES: BASELINE vs. FINE-TUNED")
print("=" * 58)
print(f"{'Metric':<12} {'Baseline':>10} {'Fine-Tuned':>12} {'Change':>10}")
print("-" * 58)

for metric in ["rouge1", "rouge2", "rougeL", "rougeLsum"]:
    base_val = baseline_results[metric]
    ft_val   = finetuned_results[metric]
    delta    = ft_val - base_val
    arrow    = "▲" if delta > 0.001 else "▼" if delta < -0.001 else "="
    print(f"{metric:<12} {base_val:>10.4f} {ft_val:>12.4f}   {arrow}{abs(delta):.4f}")

print()
print("ROUGE Quick Reference:")
print("  ROUGE-1:    Single word overlap — vocabulary")
print("  ROUGE-2:    Two-word phrase overlap — clinical phrases (most meaningful)")
print("  ROUGE-L:    Longest common subsequence — sentence structure")
print("  ROUGE-Lsum: Sentence-level ROUGE-L — best for multi-sentence impressions")
print()
print("Key question: Did ROUGE-2 improve?")
print("If yes → the model has learned radiology-specific phrase patterns.")
print("If no  → more training data or a larger model is needed.")

---
## Stretch Challenge: Tune the Hyperparameters

Try changing **one hyperparameter at a time** and observe the effect on ROUGE-2. This is how real ML practitioners develop intuition.

| Experiment | Change | Hypothesis |
|---|---|---|
| A | `learning_rate=1e-5` | Slower, more stable learning — may need more epochs |
| B | `learning_rate=1e-4` | Faster, higher risk of loss collapse — what happens at step 1? |
| C | `num_train_epochs=10` | More passes — does val loss keep improving or start rising? |
| D | `per_device_train_batch_size=4` | Noisier gradients — does it help or hurt on 850 examples? |
| E | `weight_decay=0.1` | More regularization — does it improve val loss on epoch 3+? |

**Important**: Always reload the base model before each experiment. Fine-tuning an already fine-tuned model gives confounded results.

**Reflection question**: Which hyperparameter had the biggest impact on ROUGE-2? Can you explain why based on the training loss curve?

In [ ]:
# === STRETCH CHALLENGE ===
# Uncomment and modify one hyperparameter per experiment.
# Always reload the base model first!

# from transformers import AutoModelForSeq2SeqLM
# model_exp = AutoModelForSeq2SeqLM.from_pretrained(
#     "google/flan-t5-small",
#     force_download=True,
#     dtype=torch.float32,
# ).to(device)
#
# training_args_exp = TrainingArguments(
#     output_dir="./experiment-A",
#     num_train_epochs=5,
#     per_device_train_batch_size=8,
#     learning_rate=1e-5,        # ← Change this per experiment
#     weight_decay=0.05,
#     warmup_steps=50,
#     logging_steps=25,
#     eval_strategy="epoch",
#     save_strategy="epoch",
#     load_best_model_at_end=True,
#     metric_for_best_model="eval_loss",
#     fp16=False,
#     report_to="none",
# )
#
# trainer_exp = Trainer(
#     model=model_exp,
#     args=training_args_exp,
#     train_dataset=tokenized_train,
#     eval_dataset=tokenized_val,
#     data_collator=DataCollatorForSeq2Seq(tokenizer, model=model_exp, pad_to_multiple_of=8),
# )
# trainer_exp.train()
# Then re-run the prediction and ROUGE cells using model_exp

print("Uncomment the code above to run hyperparameter experiments.")
print("Change one variable at a time and compare ROUGE-2 results.")

---
## KHCC Clinical Context and Next Steps

### What This Lab Demonstrated

| Concept | What We Observed |
|---|---|
| Fine-tuning effect | Model shifted from generic/hallucinated text to coherent radiology language |
| Loss as a training signal | Healthy loss decreased from ~9 → ~7 across 5 epochs |
| Validation monitoring | Val loss tracked train loss — no overfitting with these settings |
| ROUGE as evaluation | Quantitative, reproducible comparison of before vs. after |

### Scaling to Production at KHCC

| This Lab | Production System |
|---|---|
| 850 training examples | Full MIMIC-CXR (93K) + KHCC archive |
| flan-t5-small (80M params) | BioMedLM, Llama-3-Med, or GPT-4 fine-tuned |
| ROUGE only | ROUGE + blinded radiologist rating study (n ≥ 50) |
| English only | Arabic/English bilingual |
| Standalone script | Integrated with RIS (Radiology Information System) |
| No safety review | IRB approval, clinical risk assessment, fail-safe logic |

### Practical Takeaways for Clinical NLP

1. **Domain-specific fine-tuning works**, even with limited data — the model learned to produce radiology-appropriate language from 850 examples
2. **Debugging training requires systematic checks** — sanity check loss before training, monitor val loss during, inspect outputs qualitatively after
3. **ROUGE is a starting point**, not a final evaluation — always read the actual outputs and involve clinicians in assessment
4. **Hyperparameters matter enormously** — a 10× too-high learning rate can completely prevent learning

---
*CCI Session 4 — KHCC AI Office · King Hussein Cancer Center, Amman, Jordan*